# Ragrails — Chunking

This notebook covers:

- `chunk()` — split a directory of markdown files into RAG-ready JSON chunks
- `chunk_file()` — chunk a single file and return results in memory

Chunking takes markdown output from any ingestion method and prepares it for embedding and vector storage.

## Install

In [ ]:
# %pip install "ragrails[chunk]"

In [ ]:
from ragrails import RagRails

rag = RagRails()

---

## Chunk scraped web pages

In [ ]:
result = rag.chunk(
    input_dir="files/output/web_crawled",
    output_dir="files/output/chunks/web",
)

print("Files chunked:", result.files)
print("Total chunks:", result.chunks)
print("Output files:", result.output_files)
print("Failed:", result.failed)
print("Errors:", result.errors)

## Chunk parsed documents

In [ ]:
result = rag.chunk(
    input_dir="files/output/docs",
    output_dir="files/output/chunks/docs",
)

print("Files chunked:", result.files)
print("Total chunks:", result.chunks)

## Chunk API responses

In [ ]:
result = rag.chunk(
    input_dir="files/output/api",
    output_dir="files/output/chunks/api",
)

print("Files chunked:", result.files)
print("Total chunks:", result.chunks)

## Custom chunk settings

Use smaller chunks for tighter retrieval granularity. Use larger chunks to preserve more local context.

In [ ]:
result = rag.chunk(
    input_dir="files/output/web_crawled",
    output_dir="files/output/chunks/web",
    chunk_size=1200,
    chunk_overlap=150,
    min_chunk_length=80,
)

print("Total chunks:", result.chunks)

---

## Chunk a single file in memory

Use `chunk_file()` when you want to inspect chunks without writing to disk.

In [ ]:
chunks = rag.chunk_file(
    "files/output/web_crawled/001_index.md",
)

print("Total chunks:", len(chunks))

### Preview chunk metadata and text

In [ ]:
for chunk in chunks[:3]:
    print("Chunk ID:", chunk["metadata"]["chunk_id"])
    print("Heading:", chunk["metadata"].get("heading"))
    print("Text preview:", chunk["text"][:300])
    print("---")

### Inspect chunk metadata fields

Each chunk includes:

| Field | Description |
|---|---|
| `text` | The chunk text passed to the retriever |
| `embed_text` | The text sent to the embedding model |
| `metadata.id` | Stable document ID |
| `metadata.chunk_id` | Stable chunk ID |
| `metadata.content_hash` | Hash of chunk content |
| `metadata.heading` | Nearest markdown heading |
| `metadata.path` | Source file path |
| `metadata.title` | Document title |

In [ ]:
first = chunks[0]

print("id:", first["metadata"]["id"])
print("chunk_id:", first["metadata"]["chunk_id"])
print("content_hash:", first["metadata"]["content_hash"])
print("heading:", first["metadata"].get("heading"))
print("path:", first["metadata"].get("path"))
print("title:", first["metadata"].get("title"))

---

## Error handling

In [ ]:
result = rag.chunk(
    input_dir="files/output/web_crawled",
    output_dir="files/output/chunks/web",
)

if result.failed:
    for error in result.errors:
        print("Error:", error)